# Model Evaluation Notebook
This notebook evaluates the trained model on the test set and visualizes performance metrics.

In [ ]:
# Import Required Libraries
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve
import plotly.figure_factory as ff

# Load featurized test data
test_data = pd.read_csv("../data/bbbp_test_featurised.csv")

# Separate features and labels
X_test = test_data.drop(columns=["Y"])
y_test = test_data["Y"]

X_test = X_test.select_dtypes(include=[np.number])
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Load Trained Model
model = joblib.load("../models/bbbp_model.pkl")

# Make Predictions
y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]


In [ ]:
# Display Classification Report as a Styled Table
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=report_df.round(3).values, 
                 colLabels=report_df.columns, 
                 rowLabels=report_df.index,
                 cellLoc='center', 
                 loc='center')
plt.title("Classification Report")
plt.show()


# Interactive Classification Report Table
fig = ff.create_table(report_df.round(3))
fig.update_layout(title_text="Classification Report (Interactive Table)")
fig.show()

# Fig 1. A classification report table showing the accuracy, precision and recall values for the model, using the test data.

In [ ]:
auroc_score = roc_auc_score(y_test, y_probs)

# Compute ROC curve values
fpr, tpr, _ = roc_curve(y_test, y_probs)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'AUROC = {auroc_score:.4f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Random classifier line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend()
plt.show()

print(f"AUROC Score: {auroc_score:.4f}")

# Fig 2. The ROC Curve and AUROC Score
The Area Under the Reciever Operating Characteristic Curve(AUROC) is a metric that evaluates the ability of a classification model to correctly distinguish between positive and negative classes of it's input. It ranges from 0 to 1, with an auroc score of 1 indicating a perfect classifier.

From the graph, this model has an auroc score of **0.914**, suggesting a **high classification performance** and can be considered a reliable tool for predicting the blood brain barrier permeability of molecules.

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Class 0", "Class 1"], yticklabels=["Class 0", "Class 1"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Fig 3. Confusion Matrix

The confusion matrix shows how well the model differentiates between the two classes, and from this diagram, the model correctly predicts the permeability of **292** Class one molecules and misses only **11**(false negatives). This indicates that the model has a high sensitivity/recall. On the other hand, the model correctly predicts the impermeability for **60** molecules and misses this for **43**(false positives). The relatively high number of false positives may be due to the imbalance of the dataset, as the model had less data for the class 0 to work with. 

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.', color='purple', label='PR Curve')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()


# Fig 4. The Precision-Recall Curve

This is a visual representation of the precision and recall values seen in the the classification table, it also shows how well the model differentiates between the two classes.

In [ ]:
importances = model.feature_importances_
feature_names = X_test.columns

indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices][:10], color='green', align='center')
plt.yticks(range(10), [feature_names[i] for i in indices[:10]])
plt.xlabel("Feature Importance")
plt.title("Top 10 Important Features")
plt.gca().invert_yaxis()
plt.show()


# Fig 5. The Feature Importance Plot

This graph shows which features contributed the most to the model's decision-making process.

# Summary & Insights 📌  
🔹 The **AUROC score** of 0.9144 indicates strong classification power. 
🔹 The **confusion matrix** shows that the model performs well but struggles slightly with false positives. 
🔹 The **Precision-Recall curve** confirms the model is good at detecting positive cases while minimizing false positives.  
🔹 The **feature importance plot** reveals which molecular properties drive permeability predictions.    

🎯 **Final Thoughts:**  
This model performs well, (Is placed 4th on the TDC leaderboard for this dataset.) but further tuning and testing alternative featurisers and frameworks may improve results.
